# Day 10 — ILT 1: Implementing SCD Type 1 & Type 2 using MERGE

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Builds on** | Day 6 (dimension design — which dims are SCD1 vs SCD2 and why), Day 9 (CDF-based incremental loading) |
| **Real tables referenced** | `gbmart.bronze.customers`, `gbmart.silver.customers`, `gbmart.gold.dim_address` |
| **Duration** | 60 minutes |
| **Mode** | Instructor-led live demo — every cell below runs as-is, no setup needed |

### Learning Objectives
- Explain the difference between SCD Type 1 (overwrite, no history) and SCD Type 2 (versioned, full history)
- Understand why `MERGE` — not separate `UPDATE`/`INSERT` statements — is the right tool for both
- Walk through the real, working SCD2 pattern GlobalMart uses for `dim_customer`
- See SCD1 in contrast, using GlobalMart's `dim_address`

---
**A note on where this demo writes data:** this notebook clones the relevant tables into a scratch schema (`main.scd_demo`) instead of writing to the shared `gbmart` catalog directly. SCD-MERGE demos consume a pending change the first time they run — if this ran against the real `gbmart.silver.customers`, the next class (or the next re-run) would find nothing left to merge. Cloning keeps this demo repeatable across every cohort you teach.

## Recap — SCD Type 1 vs Type 2

You already met both types in Day 6's dimension design:

| | SCD Type 1 | SCD Type 2 |
|---|---|---|
| **What happens on change** | Overwrite the value in place | Close out the old row, insert a new one |
| **History kept?** | No — only the current value survives | Yes — every version, with `effective_start_date`/`effective_end_date` |
| **GlobalMart dims using it** | `dim_address`, `dim_payment_method` | `dim_customer`, `dim_product` |
| **Why that split** | An address correction or a payment method's display name changing isn't something you need to time-travel through | A customer's email or a product's price changing over time is exactly what `fact_sales` needs to attribute correctly — was this sale made when the product was ₹999 or ₹799? |

**The mechanical question this ILT answers: how do you actually *implement* either one, incrementally, without reprocessing the whole table every time?** The answer for both is the same SQL construct — `MERGE` — used two different ways.

## Why `MERGE`, Not Separate `UPDATE` + `INSERT`

Before `MERGE` existed, you'd write this as two separate statements:
```sql
UPDATE dim_table SET ... WHERE key IN (existing changed keys);
INSERT INTO dim_table SELECT ... WHERE key NOT IN (existing keys);
```
This has a real race-condition problem: between the `UPDATE` and the `INSERT`, another process could write to the same table, and a row that should have been updated could get inserted as a duplicate instead.

`MERGE INTO ... WHEN MATCHED ... WHEN NOT MATCHED ...` does both in **one atomic transaction** — Delta Lake guarantees the whole operation either fully succeeds or fully rolls back. This is the one operation that makes incremental dimension loading (and, in HOL 2, incremental fact loading) safe to run repeatedly without duplicate or half-applied rows.

## Setup — Clone Into a Scratch Schema

`SHALLOW CLONE` creates a new table with its own independent transaction log, but shares the underlying data files with the source — instant to create, and writes to the clone never touch the real `gbmart` tables.

In [ ]:
# ─── Scratch schema setup — isolates this demo from the real gbmart catalog ───
spark.sql("CREATE SCHEMA IF NOT EXISTS main.scd_demo")

# SHALLOW CLONE: independent transaction log, shares data files with the source.
# Writes/MERGEs against the clone below never touch gbmart.bronze/silver directly.
spark.sql("CREATE OR REPLACE TABLE main.scd_demo.bronze_customers SHALLOW CLONE gbmart.bronze.customers")
spark.sql("CREATE OR REPLACE TABLE main.scd_demo.silver_customers SHALLOW CLONE gbmart.silver.customers")
spark.sql("CREATE OR REPLACE TABLE main.scd_demo.dim_address      SHALLOW CLONE gbmart.gold.dim_address")

# CDF must be enabled on the clone independently — cloning copies data, not table properties
spark.sql("ALTER TABLE main.scd_demo.bronze_customers SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print("Scratch schema ready: main.scd_demo.{bronze_customers, silver_customers, dim_address}")

## Part 1 — SCD Type 1 in Action: `dim_address`

SCD1 is the simpler case: when an address's `address_type` label changes (say, from `"Home"` to `"Shipping"` after a customer relabels it), we don't need to know what it used to say — we just overwrite it. One `MERGE`, no `is_current`/`effective_date` bookkeeping.

In [ ]:
# Peek at one real address row we'll change, so the MERGE result is visible afterward
sample_address_id = spark.table("main.scd_demo.dim_address").select("address_id").limit(1).collect()[0][0]
spark.table("main.scd_demo.dim_address").filter(f"address_id = '{sample_address_id}'").display()

In [ ]:
from pyspark.sql import Row
from delta.tables import DeltaTable

# Simulate one incoming change: this address's type got relabeled
incoming_change_df = spark.createDataFrame([Row(address_id=sample_address_id, address_type="Shipping (Updated)")])

# SCD1 MERGE — WHEN MATCHED, just overwrite. No history columns, no versioning.
dim_address_tbl = DeltaTable.forName(spark, "main.scd_demo.dim_address")
(dim_address_tbl.alias("tgt")
    .merge(incoming_change_df.alias("src"), "tgt.address_id = src.address_id")
    .whenMatchedUpdate(set={"address_type": "src.address_type"})
    .execute()
)

print("SCD1 MERGE complete — old value is gone, only the new value survives:")
spark.table("main.scd_demo.dim_address").filter(f"address_id = '{sample_address_id}'").display()

**That's the whole pattern for SCD1.** One `MERGE`, one branch (`whenMatchedUpdate`), no way to ask "what did this used to say" afterward — which is exactly the point for a field where history has no business value.

## Part 2 — SCD Type 2 in Action: `dim_customer` (the real GlobalMart pattern)

This is the actual pattern the real pipeline uses for `dim_customer`, keyed off `email` changing. It's a 6-step flow, deliberately incremental — it never re-scans the whole customer table, only whatever changed.

In [ ]:
from pyspark.sql.functions import *

BRONZE_TABLE = "main.scd_demo.bronze_customers"
SILVER_TABLE = "main.scd_demo.silver_customers"

# Step 1 — where things stand before we isolate the incremental piece
print(f"Bronze rows: {spark.table(BRONZE_TABLE).count():,}")
print(f"Silver rows: {spark.table(SILVER_TABLE).count():,}")

In [ ]:
# Step 2 — find the version right before we'll introduce a change.
# We'll use this as LAST_PROCESSED_VERSION so CDF reads everything after it.
current_version = spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").selectExpr("max(version)").collect()[0][0]
print(f"Current Bronze version (this is LAST_PROCESSED_VERSION): {current_version}")

In [ ]:
# Simulate an incoming change: 2 real customers get a new email address.
# (In production this arrives via the normal Bronze ingestion path — for a live
# demo we apply it directly so there's a real, fresh commit for CDF to detect.)
sample_ids = [row.customer_id for row in spark.table(BRONZE_TABLE).select("CustomerID").limit(2).collect()]

for i, cid in enumerate(sample_ids):
    spark.sql(f"""
        UPDATE {BRONZE_TABLE}
        SET Email = 'updated.email.{i}@globalmart-demo.com'
        WHERE CustomerID = '{cid}'
    """)

LAST_PROCESSED_VERSION = current_version
print(f"Simulated email change for: {sample_ids}")
print(f"LAST_PROCESSED_VERSION set to {LAST_PROCESSED_VERSION} — CDF will read everything after this")

In [ ]:
# Step 3 — take ONLY the changes via CDF, not a full re-scan.
# update_preimage (the "before" row) is dropped — we only want the "after" state.
cdf_changes_df = (
    spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", LAST_PROCESSED_VERSION + 1)
        .table(BRONZE_TABLE)
        .filter("_change_type != 'update_preimage'")
)

print(f"Changed rows via CDF: {cdf_changes_df.count():,}")
cdf_changes_df.select("CustomerID", "Email", "_change_type").display()

In [ ]:
# Step 4 — process only these changed rows (same cleaning logic Day 5 used for
# the full load, just scoped down to the handful of rows that actually changed)
processed_df = cdf_changes_df \
    .filter(col("CustomerID").isNotNull() & col("Email").isNotNull()) \
    .withColumn("FirstName", trim(col("FirstName"))) \
    .withColumn("LastName", trim(col("LastName"))) \
    .withColumn("full_name", concat_ws(" ", col("FirstName"), col("LastName"))) \
    .withColumnRenamed("CustomerID", "customer_id") \
    .withColumnRenamed("Email", "email") \
    .withColumnRenamed("PhoneNumber", "phone_number") \
    .select("customer_id", "full_name", "email", "phone_number")

processed_df.display()

### Step 5 — Load into Silver: SCD2 (Close Old Version, Insert New)

**5a.** For any `customer_id` currently `is_current = true` where the incoming `email` differs — flip `is_current` to `false`, stamp `effective_end_date`. This `MERGE` only ever updates, never inserts.

**5b.** Insert every processed row as a brand-new version, `is_current = true`, fresh `customer_sk`. This covers a changed customer (new email version) — 5a already closed out their old row, so this becomes their new current one.

In [ ]:
# 5a — close out old versions where email actually changed
silver_table = DeltaTable.forName(spark, SILVER_TABLE)

(silver_table.alias("tgt")
    .merge(
        processed_df.alias("src"),
        "tgt.customer_id = src.customer_id AND tgt.is_current = true "
        "AND tgt.email <> src.email"
    )
    .whenMatchedUpdate(set={
        "is_current": "false",
        "effective_end_date": "current_date()"
    })
    .execute()
)
print("Step 5a complete — old versions closed out where email changed")

In [ ]:
# 5b — insert the new current version for each changed customer
new_versions_df = processed_df \
    .withColumn("effective_start_date", current_date()) \
    .withColumn("effective_end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True)) \
    .withColumn("customer_sk",
        sha2(concat_ws("|", col("customer_id"), col("effective_start_date").cast("string")), 256)
    ) \
    .select("customer_sk", "customer_id", "full_name", "email", "phone_number",
            "is_current", "effective_start_date", "effective_end_date")

new_versions_df.write.format("delta").mode("append").saveAsTable(SILVER_TABLE)
print(f"Step 5b complete — {new_versions_df.count()} new version row(s) inserted")

In [ ]:
# Step 6 — verify. Each changed customer_id should now show exactly 2 rows:
# old version (is_current=false, effective_end_date set) and new version (is_current=true).
spark.table(SILVER_TABLE) \
    .filter(col("customer_id").isin(sample_ids)) \
    .select("customer_sk", "customer_id", "email", "is_current", "effective_start_date", "effective_end_date") \
    .orderBy("customer_id", "effective_start_date") \
    .display()

## Recap

| | SCD1 (`dim_address`) | SCD2 (`dim_customer`) |
|---|---|---|
| **`MERGE` branches used** | `whenMatchedUpdate` only | `whenMatchedUpdate` (close out) + a separate append (new version) |
| **Rows per changed key, after** | Still 1 | 2 (old closed-out + new current) |
| **Can you answer "what was this before"?** | No | Yes — filter `is_current = false` and look at `effective_end_date` |

**What's next:** HOL 1 has you run this exact SCD2 flow yourself, end-to-end, in your own scratch schema. HOL 2 then takes the *same* MERGE technique and applies it one level up — to `fact_sales` itself, so new order lines get incrementally merged into Gold instead of a full rebuild every time.